# Знакомство с исходными таблицами `buildings` и `flats`

Исходные данные лежат в общей базе курса, доступ к ней только на чтение. Там две таблицы:

* `buildings` - дома: год постройки, тип дома, координаты, высота потолков, количество квартир и этажей, лифт;
* `flats` - квартиры: этаж, площади, количество комнат, флаги апартаментов и студии, цена и ссылка на дом.

В этом ноутбуке я смотрю, что вообще лежит в таблицах, проверяю, как они связаны,
и подбираю запрос с join, который потом переношу в шаг `extract` DAG `prepare_flats_dataset`
(файл `dags/flats_dataset.py`).

Доступы к базе берутся из файла `.env` в папке `part1_airflow`, в репозиторий он не попадает.

In [ ]:
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

## Подключаемся к общей базе

Переменные с доступами к общей базе называются `DB_SOURCE_*`. Личная база (`DB_DESTINATION_*`) здесь не нужна:
в неё пишет уже сам DAG.

In [ ]:
# load_dotenv() ищет файл .env вверх по папкам, поэтому находит part1_airflow/.env
load_dotenv()

host = os.environ['DB_SOURCE_HOST']
port = os.environ['DB_SOURCE_PORT']
user = os.environ['DB_SOURCE_USER']
password = os.environ['DB_SOURCE_PASSWORD']
db_name = os.environ['DB_SOURCE_NAME']

# quote_plus нужен, потому что в пароле бывают символы вроде @ и &, они ломают строку подключения
engine = create_engine(f'postgresql://{user}:{quote_plus(password)}@{host}:{port}/{db_name}')

## Таблица `buildings`

In [ ]:
buildings = pd.read_sql('select * from buildings', engine)
buildings.head()

In [ ]:
print('Строк и колонок:', buildings.shape)
buildings.dtypes

In [ ]:
buildings.describe()

In [ ]:
# сразу смотрю, есть ли пропуски в домах
buildings.isnull().sum()

## Таблица `flats`

`price` - это то, что потом будет предсказывать модель, а `building_id` связывает квартиру с домом.

In [ ]:
flats = pd.read_sql('select * from flats', engine)
flats.head()

In [ ]:
print('Строк и колонок:', flats.shape)
flats.dtypes

In [ ]:
flats.describe()

In [ ]:
flats.isnull().sum()

## Как связаны таблицы

Связь один-ко-многим: одному дому (`buildings.id`) соответствует много квартир (`flats.building_id`).
Проверяю два момента: сколько квартир приходится на дом и не потеряю ли я квартиры при склейке.

In [ ]:
flats_per_building = flats.groupby('building_id').size()
print('Домов, в которых есть квартиры:', flats_per_building.shape[0])
flats_per_building.describe()

In [ ]:
# квартиры, у которых нет дома с таким id: при inner join они бы потерялись
no_building = ~flats['building_id'].isin(buildings['id'])
print('Квартир без дома:', int(no_building.sum()))

# дома без квартир в датасет не попадут, и это нормально: предсказываем цену квартиры
no_flats = ~buildings['id'].isin(flats['building_id'])
print('Домов без квартир:', int(no_flats.sum()))

## Запрос, который пойдёт в DAG

Собираю квартиры вместе с характеристиками их домов одним запросом. Беру `left join`, а не `inner join`,
чтобы квартиры без дома всё равно попали в датасет: у них характеристики дома будут пустыми,
а пропуски заполняются на этапе очистки.

`f.id` переименовываю в `flat_id`, потому что в таблице-результате `flats_dataset` уже будет свой `id`.

In [ ]:
sql = '''
select
    f.id as flat_id,
    f.building_id,
    f.floor,
    f.kitchen_area,
    f.living_area,
    f.rooms,
    f.is_apartment,
    f.studio,
    f.total_area,
    f.price,
    b.build_year,
    b.building_type_int,
    b.latitude,
    b.longitude,
    b.ceiling_height,
    b.flats_count,
    b.floors_total,
    b.has_elevator
from flats as f
left join buildings as b on f.building_id = b.id
'''

dataset = pd.read_sql(sql, engine)
dataset.head()

In [ ]:
print('Размер результата:', dataset.shape)
print('Строк в flats было:', flats.shape[0])

# left join не должен ни размножить, ни потерять квартиры
assert dataset.shape[0] == flats.shape[0], 'после join изменилось число строк'
assert dataset['flat_id'].is_unique, 'flat_id повторяется'
print('flat_id уникален, значит его можно сделать ключом таблицы flats_dataset')

In [ ]:
# типы и пропуски в объединённом датасете
pd.DataFrame({'тип': dataset.dtypes, 'пропусков': dataset.isnull().sum()})

In [ ]:
dataset.sample(5, random_state=42)

## Что я выяснил

* В результате получается 18 колонок: `flat_id`, `building_id` и признаки квартиры из `flats`,
  плюс признаки дома из `buildings`. Одна строка - одна квартира.
* `flat_id` уникален, поэтому в таблице `flats_dataset` на него ставится unique-ограничение.
  Благодаря ему повторный запуск DAG не наплодит дублей: строки обновятся по этому ключу.
* В данных есть пропуски, так что колонки с целыми числами читаются как `float`,
  а булевы могут прийти как `object`. Поэтому в DAG есть отдельный шаг `transform`, который приводит типы.
* Этот же запрос стоит в шаге `extract` DAG `prepare_flats_dataset`, порядок колонок совпадает
  с порядком в шаге `create_table`.

In [ ]:
engine.dispose()